# FaceProof 08 — extensions: C2 fine-tuned ResNet + C3 frequency baseline (Day 8)

- **C2** — fine-tune ResNet-50 end-to-end (often generalizes *worse* to unseen generators).
- **C3** — DCT log-spectrum + linear SVM (Frank 2020 style frequency baseline).
Both eval in-dist + cross-gen (real paired with the fakes of interest) into the shared results.csv.

## 1. Setup (GPU on)

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q torch torchvision scikit-learn scipy joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
PROBES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

In [ ]:
import yaml, torch
from src.data import build_manifest, make_splits
df=pd.read_csv(MANIFEST)
if 'split' not in df.columns:   # ensure splits exist (self-contained)
    sources={'real':{'dir':str(CROPS_ROOT/'real'),'label':0,'generator':'real'},
             'stylegan':{'dir':str(CROPS_ROOT/'stylegan'),'label':1,'generator':'stylegan'},
             'sd':{'dir':str(CROPS_ROOT/'sd'),'label':1,'generator':'stable_diffusion'}}
    df=make_splits(build_manifest(sources), train_generators=['stylegan'], test_unseen=['stable_diffusion'], seed=13)
y=df['label'].values
fcfg=yaml.safe_load(open('configs/model.yaml'))['finetune_resnet']
DEV='cuda' if torch.cuda.is_available() else 'cpu'; print('device',DEV)
# Eval subframes: each pairs held-out reals with the fakes of interest.
real_test=df[(df['label']==0)&(df['split']=='test_indist')]
EVAL={'stylegan_indist': df[df['split']=='test_indist'],
      'stable_diffusion': pd.concat([real_test, df[df['generator']=='stable_diffusion']])}

## 2. C2 — fine-tuned ResNet-50 (train on `train`, early-stop on `val`)

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from src.metrics import auroc, eer
NORM=T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
tf=T.Compose([T.Resize((224,224)),T.ToTensor(),NORM])
class CropDS(Dataset):
    def __init__(self,sub): self.p=sub['path'].tolist(); self.y=sub['label'].tolist()
    def __len__(self): return len(self.p)
    def __getitem__(self,i): return tf(Image.open(self.p[i]).convert('RGB')), float(self.y[i])
def loader(sub,shuffle=False): return DataLoader(CropDS(sub),batch_size=fcfg['batch_size'],shuffle=shuffle,num_workers=2)
@torch.no_grad()
def score(model,sub):
    model.eval(); ps=[];ys=[]
    for xb,yb in loader(sub):
        ps.append(torch.sigmoid(model(xb.to(DEV)).squeeze(1)).cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)

In [ ]:
model=resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc=nn.Linear(model.fc.in_features,1); model=model.to(DEV)
opt=torch.optim.Adam(model.parameters(),lr=fcfg['lr']); lossf=nn.BCEWithLogitsLoss()
tr=df[df['split']=='train']; va=df[df['split']=='val']
best_auc,best_state,patience=-1.0,None,0
for ep in range(fcfg['epochs']):
    model.train()
    for xb,yb in loader(tr,shuffle=True):
        opt.zero_grad(); loss=lossf(model(xb.to(DEV)).squeeze(1),yb.to(DEV)); loss.backward(); opt.step()
    yv,pv=score(model,va); a=auroc(yv,pv); print(f'epoch {ep}: val AUROC {a:.4f}')
    if a>best_auc: best_auc,best_state,patience=a,{k:v.cpu().clone() for k,v in model.state_dict().items()},0
    else:
        patience+=1
        if patience>=fcfg['early_stopping_patience']: print('early stop'); break
model.load_state_dict(best_state); model.to(DEV)   # ensure params back on device

In [ ]:
for tg,sub in EVAL.items():
    yy,pp=score(model,sub)
    log_result(condition='c2_resnet_finetune',train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric='AUROC',value=round(auroc(yy,pp),4))
    log_result(condition='c2_resnet_finetune',train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric='EER',value=round(eer(yy,pp),4))
    print(f'C2 {tg}: AUROC {auroc(yy,pp):.4f}')
torch.save(model.state_dict(), PROBES_ROOT/'c2_resnet_finetune.pt')

## 3. C3 — DCT log-spectrum + linear SVM

In [ ]:
from src.leakage_check import dct_log_features
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def dct_matrix(sub):
    feats=[dct_log_features(p) for p in sub['path']]
    keep=np.array([f is not None for f in feats])
    return np.array([f for f in feats if f is not None]), sub['label'].values[keep]
Xtr,ytr=dct_matrix(df[df['split']=='train'])
svm=make_pipeline(StandardScaler(),LinearSVC(max_iter=5000)).fit(Xtr,ytr)
for tg,sub in EVAL.items():
    Xte,yte=dct_matrix(sub); s=svm.decision_function(Xte)
    log_result(condition='c3_frequency',train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric='AUROC',value=round(auroc(yte,s),4))
    log_result(condition='c3_frequency',train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric='EER',value=round(eer(yte,s),4))
    print(f'C3 {tg}: AUROC {auroc(yte,s):.4f}')
print('✅ Day 8 gate: all four conditions (C1-C4) in results.csv')